# JRC hazards — catalog explorer (no network)

The `earthlens.jrc` backend serves every JRC / Copernicus-EMS hazard product from
one class, selected by dataset and dispatched on the catalog row's `kind`:

- **`flood_hazard_raster`** — the European Flood Hazard Map (EFHM): river-flood
  **water depth (m)** for a set of **return periods** over Europe and the
  Mediterranean.
- **`sea_level_gridded`** — the probabilistic **Total Water Level** forecasts
  (medium-term and subseasonal), global 0.25° NetCDF cubes.
- **`sea_level_coastal`** — the subseasonal global per-country coastal summary.

This notebook inspects the catalog and the URL / pixel-window helpers without
touching the network.

## The catalog

Four datasets across the three kinds. The EFHM is addressed by return period; the
sea-level rows are addressed by forecast cycle.

In [ ]:
from earthlens.jrc import Catalog

catalog = Catalog()
for name in sorted(catalog.datasets):
    row = catalog.get(name)
    print(f"{name:32s} kind={row.kind:22s} units={row.units or '-':4s} crs={row.crs}")

print()
efhm = catalog.get("efhm")
print("efhm return periods:", efhm.return_periods)
sea = catalog.get("sea_level_medium_term")
print(
    "medium-term:",
    sea.cadence,
    "| horizon:",
    sea.horizon_days,
    "days",
    "| default field:",
    sea.default_field,
)
print("licence:", catalog.license_id)

## How a small AOI stays cheap

Each return period is one whole-Europe GeoTIFF (~23 GB uncompressed). The backend
never reads it whole: it opens the file lazily and reads only the AOI's pixel
window over `/vsicurl` (HTTP range requests) via pyramids' `Dataset.crop(bbox=)`.
Building the request — the per-return-period URL below — is offline; only
`.download()` touches the network.

In [ ]:
from earthlens.jrc._helpers import efhm_url

# One whole-Europe ~23 GB GeoTIFF per return period; a small AOI reads only its
# pixel window via pyramids' Dataset.crop(bbox=) over /vsicurl (a few hundred KB).
for rp in (100, 200, 500):
    print(f"RP{rp}:", efhm_url(rp))

## Next

See **EFHM quickstart** for a real windowed download and map, and
**Sea-level TWL forecast** for the coastal forecast products.